In [1]:
import os
import cv2
import math
import xml.etree.ElementTree as ET
from xml.dom import minidom
from ultralytics import YOLO
from tqdm.notebook import tqdm
import numpy as np

In [2]:
def prettify_xml(elem):
    """Return a pretty-printed XML string for the Element."""
    rough_string = ET.tostring(elem, 'utf-8')
    reparsed = minidom.parseString(rough_string)
    return reparsed.toprettyxml(indent="  ")

In [3]:
def create_cvat_xml(image_predictions_map, output_xml_path):
    """
    Creates a CVAT compatible XML file from predictions using <box> for rotated bounding boxes.

    Args:
        image_predictions_map (dict): A dictionary where keys are image filenames
                                      and values are dicts containing 'dimensions' (w,h)
                                      and 'predictions' list. Each prediction is a
                                      tuple (label, confidence, xtl, ytl, xbr, ybr, rotation_deg_cvat).
        output_xml_path (str): Path to save the output XML file.
    """
    annotations_node = ET.Element("annotations")
    ET.SubElement(annotations_node, "version").text = "1.1"

    meta_node = ET.SubElement(annotations_node, "meta")
    task_node = ET.SubElement(meta_node, "task")
    ET.SubElement(task_node, "mode").text = "annotation"
    labels_node = ET.SubElement(task_node, "labels")
    unique_labels = set()
    for data in image_predictions_map.values():
        for pred_tuple in data['predictions']: # pred_tuple[0] is the label
            unique_labels.add(pred_tuple[0]) 
    for label_name in sorted(list(unique_labels)):
        label_el = ET.SubElement(labels_node, "label")
        ET.SubElement(label_el, "name").text = label_name

    sorted_image_names = sorted(image_predictions_map.keys())

    for image_id, image_name in enumerate(sorted_image_names):
        data = image_predictions_map[image_name]
        img_width, img_height = data['dimensions']
        predictions = data['predictions']

        image_node = ET.SubElement(annotations_node, "image", id=str(image_id),
                                   name=image_name,
                                   width=str(img_width), height=str(img_height))

        # Unpack the new prediction data structure
        for label, confidence, xtl, ytl, xbr, ybr, rotation_deg_cvat in predictions:
            # Create a <box> element for rotated bounding box
            box_node = ET.SubElement(image_node, "box",
                                     label=label,
                                     occluded="0",
                                     source="manual", # Or "automatic" / "semi-automatic"
                                     # Attributes for rotated bounding box
                                     xtl=f"{xtl:.2f}",
                                     ytl=f"{ytl:.2f}",
                                     xbr=f"{xbr:.2f}",
                                     ybr=f"{ybr:.2f}",
                                     rotation=f"{rotation_deg_cvat:.2f}", # Clockwise rotation in degrees
                                     z_order="0"
                                     )
            # Add confidence as an attribute
            attribute_node = ET.SubElement(box_node, "attribute", name="confidence")
            attribute_node.text = f"{confidence:.2f}"

    xml_str = prettify_xml(annotations_node)
    with open(output_xml_path, "w", encoding='utf-8') as f:
        f.write(xml_str)
    print(f"CVAT XML annotations saved to: {output_xml_path}")

In [9]:
def parse_existing_prediction_txt(txt_path, class_names_map, image_width, image_height):
    """
    Parses an existing prediction TXT file.
    Each line: <class_id> <x1> <y1> <x2> <y2> <x3> <y3> <x4> <y4>

    Args:
        txt_path (str): Path to the TXT prediction file.
        class_names_map (dict): Dictionary mapping class_id (int) to class_name (str).

    Returns:
        list: A list of prediction tuples in CVAT format:
              (label, confidence, xtl, ytl, xbr, ybr, rotation_deg_cvat)
    """
    predictions = []
    if not os.path.exists(txt_path):
        return predictions

    with open(txt_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 9:  # class_id + 8 coordinates
                print(f"Warning: Malformed line in {txt_path}: {line.strip()}. Skipping.")
                continue
            try:
                class_id = int(parts[0])
                coords = [float(c) for c in parts[1:]]
            except ValueError:
                print(f"Warning: Non-numeric data in {txt_path}: {line.strip()}. Skipping.")
                continue

            label = class_names_map.get(class_id, f"unknown_class_{class_id}")

            # Points for minAreaRect: (N, 1, 2) or (N,2)
            points = np.array([
                [coords[0] * image_width, coords[1] * image_height],
                [coords[2] * image_width, coords[3] * image_height],
                [coords[4] * image_width, coords[5] * image_height],
                [coords[6] * image_width, coords[7] * image_height]
            ], dtype=np.float32)

            # Get OBB parameters using minAreaRect
            # rect is: ((center_x, center_y), (width, height), angle_opencv)
            # angle_opencv is in degrees, range [-90, 0]
            # width and height might be swapped, angle reflects this.
            rect = cv2.minAreaRect(points)
            
            center_x, center_y = rect[0]
            # minAreaRect might return width < height or width > height.
            # The angle is the rotation of the 'width' side (rect[1][0]) from the x-axis.
            # For CVAT, xtl,ytl,xbr,ybr define the UNROTATED box.
            # So, rect[1][0] and rect[1][1] are the dimensions for this unrotated box.
            box_width, box_height = rect[1]
            angle_opencv = rect[2] # Typically in [-90, 0] degrees

            xtl = center_x - box_width / 2
            ytl = center_y - box_height / 2
            xbr = center_x + box_width / 2
            ybr = center_y + box_height / 2

            # CVAT rotation: clockwise, degrees, in [0, 360)
            # OpenCV angle: If -30, it means the 'width' edge is 30 deg CW from horizontal.
            # So, CVAT rotation should be 30. Thus, -angle_opencv.
            cvat_rotation_deg = (angle_opencv) % 360.0
            # Ensure it's always positive (Python's % handles negative results well for this purpose)
            # e.g., -(-80) % 360 = 80.  -(-10) % 360 = 10.

            confidence = 1.0  # No confidence in TXT files, assume 1.0 (perfect)
            
            predictions.append((label, confidence, xtl, ytl, xbr, ybr, cvat_rotation_deg))
    return predictions

In [11]:
def yolo_obb_predict_to_cvat(model_path, image_dir, output_xml, conf_threshold=0.25, existing_predictions_dir=None):
    """
    Performs YOLO-OBB prediction or loads existing predictions for all images in a directory
    and saves results to CVAT XML using <box> tags for oriented bounding boxes.

    Args:
        model_path (str): Path to the YOLO-OBB .pt model file.
        image_dir (str): Path to the directory containing images.
        output_xml (str): Path to save the output CVAT XML file.
        conf_threshold (float): Confidence threshold for YOLO predictions.
        existing_predictions_dir (str, optional): Path to directory with existing .txt predictions.
                                                  If None, YOLO prediction is always performed.
    """
    if not os.path.exists(model_path):
        print(f"Error: Model path '{model_path}' does not exist.")
        return
    if not os.path.isdir(image_dir):
        print(f"Error: Image directory '{image_dir}' does not exist.")
        return
    if existing_predictions_dir and not os.path.isdir(existing_predictions_dir):
        print(f"Warning: Existing predictions directory '{existing_predictions_dir}' does not exist. Will proceed with YOLO predictions.")
        existing_predictions_dir = None # Disable if path is invalid

    try:
        print(f"Loading model from: {model_path}")
        model = YOLO(model_path)
        if not hasattr(model, 'obb') and not model.task == 'obb':
             print(f"Warning: Model '{model_path}' might not be an OBB model or task not set to 'obb'. Ensure it's trained for OBB.")
    except Exception as e:
        print(f"Error loading YOLO model: {e}")
        return

    class_names = model.names # Needed for both YOLO and parsing TXT
    image_predictions_map = {}
    supported_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')

    image_files = [f for f in os.listdir(image_dir) if f.lower().endswith(supported_extensions)]
    if not image_files:
        print(f"No supported image files found in '{image_dir}'.")
        return

    print(f"Processing {len(image_files)} images from '{image_dir}'...")
    for image_file in tqdm(image_files, desc="Processing images"):
        image_path = os.path.join(image_dir, image_file)
        
        try:
            img = cv2.imread(image_path)
            if img is None:
                print(f"Warning: Could not read image '{image_path}'. Skipping.")
                continue
            img_height, img_width = img.shape[:2]

            current_image_predictions = []
            parsed_from_file = False

            # Check for existing predictions
            if existing_predictions_dir:
                base_name, _ = os.path.splitext(image_file)
                txt_pred_path = os.path.join(existing_predictions_dir, base_name + ".txt")
                if os.path.exists(txt_pred_path):
                    # tqdm.write(f"Loading existing predictions for {image_file} from {txt_pred_path}")
                    current_image_predictions = parse_existing_prediction_txt(txt_pred_path, class_names, img_width, img_height)
                    parsed_from_file = True
            
            # If no existing predictions found, run YOLO
            if not parsed_from_file:
                # tqdm.write(f"Running YOLO prediction for {image_file}")
                results = model.predict(image_path, conf=conf_threshold, verbose=False, task='obb')

                if results and results[0].obb is not None:
                    obb_predictions = results[0].obb
                    for i in range(len(obb_predictions.cls)):
                        class_id = int(obb_predictions.cls[i])
                        label = class_names[class_id]
                        confidence = float(obb_predictions.conf[i])

                        obb_xywhr_data = obb_predictions.xywhr[i]
                        cx, cy, w, h = obb_xywhr_data[0].item(), obb_xywhr_data[1].item(), obb_xywhr_data[2].item(), obb_xywhr_data[3].item()
                        angle_rad_ultralytics = obb_xywhr_data[4].item()
                        
                        xtl = cx - w / 2
                        ytl = cy - h / 2
                        xbr = cx + w / 2
                        ybr = cy + h / 2

                        # CVAT rotation: clockwise, degrees, in [0, 360)
                        # Ultralytics angle: radians, typically CCW positive.
                        # To convert to CVAT's CW positive: negate and convert to degrees.
                        angle_deg_ultralytics_ccw = math.degrees(angle_rad_ultralytics)
                        cvat_rotation_deg = (angle_deg_ultralytics_ccw) % 360.0
                        
                        current_image_predictions.append((label, confidence, xtl, ytl, xbr, ybr, cvat_rotation_deg))
                elif results and results[0].boxes is not None and len(results[0].boxes) > 0:
                     print(f"Warning: Model seems to have produced standard bounding boxes (not OBB) for {image_file}. Ensure your model is an OBB model and the task is correctly inferred or set.")

            image_predictions_map[image_file] = {
                'dimensions': (img_width, img_height),
                'predictions': current_image_predictions
            }

        except Exception as e:
            print(f"Error processing image '{image_path}': {e}")
            import traceback
            traceback.print_exc()
            continue

    if not image_predictions_map:
        print("No predictions were made or loaded for any image, or no images were processed.")
        return

    create_cvat_xml(image_predictions_map, output_xml)

In [12]:
# --- Ячейка с конфигурацией ---
MODEL_PATH = "runs/obb/120/train/weights/best.pt"
IMAGE_DIR = "norm_images/"
OUTPUT_XML_PATH = "datasets/pretrained/obb_predictions_cvat_v2.xml" # Новое имя файла, чтобы не перезаписать старый
CONFIDENCE_THRESHOLD = 0.35

# --- НОВЫЙ ПАРАМЕТР ---
# Укажите путь к папке, где лежат ваши .txt файлы с предсказаниями
# Например, "path/to/my/text_predictions/"
# Если такой папки нет или вы не хотите использовать существующие предсказания, установите в None
EXISTING_PREDICTIONS_DIR = "datasets/120_annotations/labels/train/" # <--- ИЗМЕНИТЕ ЭТОТ ПУТЬ

# --- Sanity Checks ---
all_paths_ok = True
if not os.path.exists(MODEL_PATH):
    print(f"ERROR: Model file not found at '{MODEL_PATH}'. Please check the path.")
    all_paths_ok = False
if not os.path.isdir(IMAGE_DIR):
    print(f"ERROR: Image directory not found at '{IMAGE_DIR}'. Please check the path.")
    all_paths_ok = False
if EXISTING_PREDICTIONS_DIR and not os.path.isdir(EXISTING_PREDICTIONS_DIR):
    print(f"WARNING: Existing predictions directory not found at '{EXISTING_PREDICTIONS_DIR}'. Will proceed with YOLO for all images.")
    # EXISTING_PREDICTIONS_DIR = None # Можно раскомментировать, если хотите автоматически отключать при неверном пути

if all_paths_ok:
    print("Configuration seems OK. Ready to run.")
else:
    print("Please correct the paths in the configuration.")

# MODEL_PATH = "runs/obb/120/train/weights/best.pt"
# IMAGE_DIR = "norm_images/"
# OUTPUT_XML_PATH = "datasets/pretrained/obb_predictions_cvat.xml"

# # Confidence threshold for predictions (0.0 to 1.0)
# CONFIDENCE_THRESHOLD = 0.35

# # --- Sanity Checks (Optional but Recommended) ---
# if not os.path.exists(MODEL_PATH):
#     print(f"ERROR: Model file not found at '{MODEL_PATH}'. Please check the path.")
# elif not os.path.isdir(IMAGE_DIR):
#     print(f"ERROR: Image directory not found at '{IMAGE_DIR}'. Please check the path.")
# else:
#     print("Configuration seems OK. Ready to run.")
#     # You might want to download a sample OBB model if you don't have one
#     # For example, from ultralytics:
#     # if MODEL_PATH == "yolov8n-obb.pt" and not os.path.exists("yolov8n-obb.pt"):
#     #     print("Downloading yolov8n-obb.pt...")
#     #     YOLO("yolov8n-obb.pt") # This will download it if not present
#     #     print("Model downloaded.")

Configuration seems OK. Ready to run.


In [13]:
if all_paths_ok: # Проверяем, что основные пути корректны
    yolo_obb_predict_to_cvat(
        model_path=MODEL_PATH,
        image_dir=IMAGE_DIR,
        output_xml=OUTPUT_XML_PATH,
        conf_threshold=CONFIDENCE_THRESHOLD,
        existing_predictions_dir=EXISTING_PREDICTIONS_DIR # <--- Новый аргумент
    )
else:
    print("Please correct the MODEL_PATH and IMAGE_DIR in the configuration cell before running this cell.")

# if os.path.exists(MODEL_PATH) and os.path.isdir(IMAGE_DIR):
#     yolo_obb_predict_to_cvat(
#         model_path=MODEL_PATH,
#         image_dir=IMAGE_DIR,
#         output_xml=OUTPUT_XML_PATH,
#         conf_threshold=CONFIDENCE_THRESHOLD
#     )
# else:
#     print("Please correct the MODEL_PATH and IMAGE_DIR in Cell 5 before running this cell.")

Loading model from: runs/obb/120/train/weights/best.pt
Processing 599 images from 'norm_images/'...


Processing images:   0%|          | 0/599 [00:00<?, ?it/s]

CVAT XML annotations saved to: datasets/pretrained/obb_predictions_cvat_v2.xml
